In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
import torch.nn as nn
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression

hits = glob.glob("/kaggle/input/**/book_train.parquet", recursive=True)
if hits:
    data_dir = os.path.dirname(hits[0]) + "/"
elif os.path.exists("../../data/book_train.parquet"):
    data_dir = "../../data/"
else:
    os.system("ls -R /kaggle/input | head -50")
    raise FileNotFoundError("没找到比赛数据，检查右侧 Input 有没有加上 optiver-realized-volatility-prediction")
print(data_dir)

In [ ]:
def realized_vol(series):
    return np.sqrt(np.sum(np.square(series)))

def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

def feval_rmspe(preds, data):
    return "rmspe", rmspe(data.get_label(), preds), False

def window_stats(df, col, keys=["stock_id", "time_id"], prefix = ""):
    g = df.groupby(keys, observed=True)[col]
    out = pd.DataFrame({
        "mean": g.mean(),
        "std":  g.std(ddof=0),
        "last": g.last(),
        "max": g.max(),
    })
    out["zscore_last"] = ((out["last"] - out["mean"]) / out["std"]).where(out["std"] > 0, 0.0)
    return out.drop(columns="last").add_prefix(f"{prefix}{col}_")

def add_market_features(df, transform, stats=("mean",), rel=True):
    fn = {"log": np.log, "log1p": np.log1p, "logc": lambda x: np.log(x.clip(lower=1e-6))}
    out = {}
    for c, how in transform.items():
        x = fn[how](df[c]) if how else df[c]
        name = f"{how}_{c}" if how else c
        g = x.groupby(df["time_id"])
        for s in stats:
            out[f"mkt_{s}_{name}"] = g.transform(s)
        if rel:
            out[f"rel_{name}"] = x - g.transform("mean")
    return pd.concat([df, pd.DataFrame(out, index=df.index)], axis=1)

In [ ]:
def prepare(book, trade):
    book["stock_id"] = book["stock_id"].astype("int16")
    trade["stock_id"] = trade["stock_id"].astype("int16")

    # total book size
    book["tot_size"] = book["bid_size1"] + book["bid_size2"] + book["ask_size1"] + book["ask_size2"]

    # WAP (3 versions)
    book["wap1"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
                 / (book["bid_size1"] + book["ask_size1"])).astype("float32")
    book["wap2"] = ((book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
                 / (book["bid_size2"] + book["ask_size2"])).astype("float32")
    book["wap12"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"] +
                book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
                 / book["tot_size"]).astype("float32")

    log_cols = ["log_wap1", "log_wap2", "log_wap12"]
    ret_cols = ["log_ret1", "log_ret2", "log_ret12"]
    for k in ["1", "2", "12"]:
        book[f"log_wap{k}"] = np.log(book[f"wap{k}"])
    book[ret_cols] = book.groupby(["stock_id", "time_id"], observed=True)[log_cols].diff().to_numpy()
    book.drop(columns=log_cols, inplace=True)

    # spread
    book["spread"] = book["ask_price1"] - book["bid_price1"]

    # imbalance
    book["imb1"] = ((book.bid_size1 - book.ask_size1) / (book.bid_size1 + book.ask_size1)).astype("float32")
    book["imb12"] = ((book.bid_size1 + book.bid_size2 - book.ask_size1 - book.ask_size2)
                     / (book.bid_size1 + book.bid_size2 + book.ask_size1 + book.ask_size2)).astype("float32")
    book["imb1_abs"] = book["imb1"].abs()
    book["imb12_abs"] = book["imb12"].abs()

    # trade log return
    trade["log_price"] = np.log(trade["price"])
    trade["log_ret"] = trade.groupby(["stock_id", "time_id"])["log_price"].diff()

    # buy/sell, inferred from the quote before the trade
    bid = book["bid_price1"].to_numpy()
    ask = book["ask_price1"].to_numpy()
    mid = (bid + ask) / 2
    hs = (ask - bid) / 2

    sid = book["stock_id"].to_numpy()
    tid = book["time_id"].to_numpy()
    secs = book["seconds_in_bucket"].to_numpy()
    same_window = np.r_[False, (sid[1:] == sid[:-1]) & (tid[1:] == tid[:-1])]

    n_windows = len(pd.MultiIndex.from_arrays([sid, tid]).unique())
    assert (~same_window).sum() == n_windows, "有窗口的行不连续"
    assert (np.diff(secs)[same_window[1:]] > 0).all(), "窗口内秒数不是递增的"

    mid_prev = np.where(same_window, np.roll(mid, 1), mid)
    hs_prev = np.where(same_window, np.roll(hs, 1), hs)

    def row_key(df):
        return (df["stock_id"].to_numpy().astype(np.int64) * 100_000_000
                + df["time_id"].to_numpy().astype(np.int64) * 1_000
                + df["seconds_in_bucket"].to_numpy())

    pos = pd.Index(row_key(book)).get_indexer(row_key(trade))

    tq = trade[["stock_id", "time_id", "seconds_in_bucket", "price", "size"]].copy()
    x = (tq["price"].to_numpy() - mid_prev[pos]) / hs_prev[pos]
    tq["x"] = np.where(pos >= 0, x, np.nan)

    THRESH = 0.5
    tq["is_buy"] = tq["x"] >= THRESH
    tq["is_sell"] = tq["x"] <= -THRESH
    tq["buy_vol"] = tq["size"] * tq["is_buy"]
    tq["sell_vol"] = tq["size"] * tq["is_sell"]
    tq["abs_x"] = tq["x"].abs()
    tq["sweep"] = tq["abs_x"] > 1.5
    return tq


def window_features(book, trade, tq):
    grouped = book.groupby(["stock_id", "time_id"], observed=True)
    trade_grouped = trade.groupby(["stock_id", "time_id"], observed=True)

    imb_features = grouped.agg(
        imb1_mean     =("imb1", "mean"),
        imb1_std      =("imb1", "std"),
        imb1_abs_mean =("imb1_abs", "mean"),
        imb12_mean    =("imb12", "mean"),
        imb12_std     =("imb12", "std"),
        imb12_abs_mean=("imb12_abs", "mean"),
    )

    # first and last wap distance to 1
    fwap1_to1 = (grouped["wap1"].first() - 1.).abs().rename("fwap1_to1")
    lwap1_to1 = (grouped["wap1"].last() - 1.).abs().rename("lwap1_to1")

    # book realized vol (3 versions)
    book_rv = grouped[["log_ret1", "log_ret2", "log_ret12"]].agg(realized_vol)
    book_rv.rename(columns={"log_ret1":"rv1", "log_ret2":"rv2", "log_ret12":"rv12"}, inplace=True)

    # trade realized vol
    trade_rv = trade_grouped[["log_ret"]].agg(realized_vol)
    trade_rv.rename(columns={"log_ret":"trade_rv"}, inplace=True)

    # trade ratio
    n_quotes = grouped.size().rename("n_quotes")
    n_trades = trade_grouped.size().rename("n_trades")
    trade_ratio = pd.merge(n_quotes, n_trades, on = ["stock_id", "time_id"], how = "left")
    trade_ratio["trade_ratio"] = trade_ratio["n_trades"] / trade_ratio["n_quotes"]
    trade_ratio.fillna(0, inplace = True)

    # trade total volume
    trade_tot_volume = trade_grouped["size"].sum().rename("trade_tot_volume")

    side_feats = tq.groupby(["stock_id", "time_id"], observed=True).agg(
        n_buy=("is_buy", "sum"),
        n_sell=("is_sell", "sum"),
        buy_vol=("buy_vol", "sum"),
        sell_vol=("sell_vol", "sum"),
        abs_x_mean=("abs_x", "mean"),
        sweep_frac=("sweep", "mean"),
    )
    side_feats["buy_frac"] = side_feats["n_buy"] / (side_feats["n_buy"] + side_feats["n_sell"])
    side_feats["buy_vol_frac"] = side_feats["buy_vol"] / (side_feats["buy_vol"] + side_feats["sell_vol"])
    side_feats = side_feats[["buy_frac", "buy_vol_frac", "abs_x_mean", "sweep_frac"]]

    features = []
    for feat in ["spread", "wap1", "wap2", "wap12", "tot_size", "log_ret1", "log_ret12"]:
        features.append(window_stats(book, feat))
    for feat in ["size", "price", "log_ret"]:
        features.append(window_stats(trade, feat, prefix = "trade_"))
    features.extend([imb_features, fwap1_to1, lwap1_to1, book_rv, trade_ratio, trade_rv, trade_tot_volume, side_feats])
    return pd.concat(features, axis = 1)


SLICES = [300, 450]
COUNT_COLS = ["n_quotes", "n_trades", "trade_ratio"]


def make_features(book, trade):
    tq = prepare(book, trade)
    out = [window_features(book, trade, tq)]
    for s in SLICES:
        b = book[book["seconds_in_bucket"] >= s]
        t = trade[trade["seconds_in_bucket"] >= s]
        q = tq[tq["seconds_in_bucket"] >= s]
        out.append(window_features(b, t, q).add_suffix(f"_{s}"))
    features_df = pd.concat(out, axis = 1)
    cnt = [f"{c}_{s}" for s in SLICES for c in COUNT_COLS]
    features_df[cnt] = features_df[cnt].fillna(0)
    return features_df.reset_index()


def build_features(book_dir, trade_dir):
    out = []
    for p in sorted(glob.glob(book_dir + "stock_id=*")):
        sid = int(p.split("=")[-1])
        book = pd.read_parquet(p)
        book["stock_id"] = sid
        tp = trade_dir + f"stock_id={sid}"
        if os.path.exists(tp):
            trade = pd.read_parquet(tp)
        else:
            trade = pd.DataFrame({"time_id": pd.Series(dtype="int16"), "seconds_in_bucket": pd.Series(dtype="int16"),
                                  "price": pd.Series(dtype="float32"), "size": pd.Series(dtype="int32"),
                                  "order_count": pd.Series(dtype="int16")})
        trade["stock_id"] = sid
        out.append(make_features(book, trade))
    features_df = pd.concat(out, ignore_index=True)

    transform = {"rv1": "log", "rv2": "log", "trade_rv": "log1p", "n_quotes": "log", "n_trades": "log1p"}
    for s in SLICES:
        transform.update({f"rv1_{s}": "logc", f"rv2_{s}": "logc", f"trade_rv_{s}": "log1p",
                          f"n_quotes_{s}": "log1p", f"n_trades_{s}": "log1p"})
    return add_market_features(features_df, transform, stats=("mean", "std"))

In [ ]:
train_features = build_features(data_dir + "book_train.parquet/", data_dir + "trade_train.parquet/")
test_features = build_features(data_dir + "book_test.parquet/", data_dir + "trade_test.parquet/")

target = pd.read_csv(data_dir + "train.csv")
df = target.merge(train_features, on=["stock_id", "time_id"], how="left", validate="one_to_one")
test = pd.read_csv(data_dir + "test.csv")
test_df = test.merge(test_features, on=["stock_id", "time_id"], how="left", validate="one_to_one")

feature_cols = [c for c in train_features.columns if c != "time_id"]

X = df[feature_cols]
y = df["target"].to_numpy()
w = 1 / y ** 2

In [ ]:
folds = list(GroupKFold(n_splits=5, shuffle=True, random_state=0).split(X, y, groups=df["time_id"]))

params = {
    "objective": "rmse",
    "metric": "None",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 500,
    "feature_fraction": 0.5,
    "bagging_fraction": 0.72,
    "bagging_freq": 4,
    "max_bin": 100,
    "lambda_l1": 0.5,
    "lambda_l2": 1.0,
    "seed": 0,
    "verbose": -1,
}

data = lgb.Dataset(X, y, weight=w, categorical_feature=["stock_id"])

results = lgb.cv(
    params, data, num_boost_round=3000, folds=folds, feval=feval_rmspe,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
    return_cvbooster=True,
)

print(len(results["valid rmspe-mean"]), results["valid rmspe-mean"][-1])

cvbooster = results["cvbooster"]
oof_lgb = np.zeros(len(y))
for (tr, va), b in zip(folds, cvbooster.boosters):
    oof_lgb[va] = b.predict(X.iloc[va], num_iteration=cvbooster.best_iteration)
pred_lgb = np.mean(cvbooster.predict(test_df[feature_cols], num_iteration=cvbooster.best_iteration), axis=0)
print("LGB OOF RMSPE:", rmspe(y, oof_lgb))

In [ ]:
num_cols = [c for c in feature_cols if c != "stock_id"]
Z = df[num_cols].replace([np.inf, -np.inf], np.nan)
Zt = test_df[num_cols].replace([np.inf, -np.inf], np.nan)
log_cols = [c for c in num_cols if Z[c].min() > 0 and Z[c].skew() > 2]
Z[log_cols] = np.log(Z[log_cols])
Zt[log_cols] = np.log(Zt[log_cols].clip(lower=Z[log_cols].min(), axis=1))

sid = df["stock_id"].to_numpy()
sid_test = test_df["stock_id"].to_numpy()
dev = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


class MLP(nn.Module):
    def __init__(self, n_num, n_stocks=127, emb_dim=24, hidden=(128, 64, 32)):
        super().__init__()
        self.emb = nn.Embedding(n_stocks, emb_dim)
        layers, d = [], n_num + emb_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.SiLU()]
            d = h
        self.mlp = nn.Sequential(*layers)
        self.out = nn.Linear(d, 1)

    def forward(self, x, s):
        return torch.exp(self.out(self.mlp(torch.cat([x, self.emb(s)], 1)))).squeeze(1)


def rmspe_t(p, t):
    return torch.sqrt(torch.mean(((t - p) / t) ** 2))


def to_t(a, dtype=torch.float32):
    return torch.tensor(a, dtype=dtype, device=dev)


oof_nn = np.zeros(len(y))
pred_nn = np.zeros(len(test_df))

for k, (tr, va) in enumerate(folds):
    torch.manual_seed(k)
    med = Z.iloc[tr].median()
    A, At = Z.fillna(med), Zt.fillna(med)
    mu, sd = A.iloc[tr].mean(), A.iloc[tr].std().replace(0, 1)
    A = ((A - mu) / sd).clip(-5, 5).to_numpy(np.float32)
    At = ((At - mu) / sd).clip(-5, 5).to_numpy(np.float32)

    xt, st, yt = to_t(A[tr]), to_t(sid[tr], torch.long), to_t(y[tr])
    xv, sv, yv = to_t(A[va]), to_t(sid[va], torch.long), to_t(y[va])

    model = MLP(A.shape[1]).to(dev)
    with torch.no_grad():
        model.out.bias.fill_(float(np.log(y[tr]).mean()))
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.2, patience=4)

    best, bad = 1e9, 0
    for ep in range(200):
        model.train()
        perm = torch.randperm(len(tr), device=dev)
        for i in range(0, len(tr), 2048):
            b = perm[i:i + 2048]
            opt.zero_grad()
            rmspe_t(model(xt[b], st[b]), yt[b]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            v = rmspe_t(model(xv, sv), yv).item()
        sched.step(v)
        if v < best - 1e-5:
            best, bad = v, 0
            best_state = {kk: vv.clone() for kk, vv in model.state_dict().items()}
        else:
            bad += 1
            if bad >= 12:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        oof_nn[va] = model(xv, sv).cpu().numpy()
        pred_nn += model(to_t(At), to_t(sid_test, torch.long)).cpu().numpy() / len(folds)
    print(f"NN fold {k}: {rmspe(y[va], oof_nn[va]):.4f}  epochs {ep + 1}")

print("NN OOF RMSPE:", rmspe(y, oof_nn))

In [ ]:
S = np.column_stack([np.log(oof_lgb), np.log(oof_nn)])
S_test = np.column_stack([np.log(pred_lgb), np.log(pred_nn)])

meta = LinearRegression().fit(S, np.log(y))
p = np.exp(meta.predict(S))
c = np.sum(p / y) / np.sum((p / y) ** 2)
print("meta coef:", meta.coef_, "c:", c, "stacked OOF RMSPE:", rmspe(y, c * p))

test_df["target"] = c * np.exp(meta.predict(S_test))
test_df[["row_id", "target"]].to_csv("submission.csv", index=False)
test_df[["row_id", "target"]].head()